# 01 - LLM Bootstrap Labelling

Uses an LLM (Gemini API) to **bootstrap** the HITL classifier seed labels.
Replaces — or precedes — the human seed step described in `classification_strategy.md` (Step 0).

**Inputs:** `llm_bootstrap_dataset.pkl` (~10 000 tweets carved out by `00_hitl_data_preparation.ipynb`).
This subset is disjoint from `base_dataset.pkl`, the HITL batches, and `inference_dataset.pkl` —
see `partition_ids.pkl` for the manifest.

**Pipeline:**
1. Load `llm_bootstrap_dataset.pkl`.
2. For each tweet, call the LLM with a prompt containing the per-category criteria.
3. Parse the JSON response, validate the label, retry on transient errors.
4. Checkpoint every N tweets to survive Colab disconnects.
5. Save the final CSV with the **same schema** as `hitl_review_batch_*.csv`
   so it drops straight into `02_hitl_training_loop.ipynb`.

Two cells are **placeholders** that must be filled in before running:
the category list + criteria, and the Gemini API key + model id.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'google-genai'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import json
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm
from google import genai
from google.genai import types

## Configuration

Tune the constants below per run. `SMOKE_TEST` keeps the pipeline cheap during development.

In [ ]:
# ── Run mode ────────────────────────────────────────────────────────────
SMOKE_TEST   = True       # True → label SMOKE_TEST_N tweets only; flip to False for the full run
SMOKE_TEST_N = 100

# ── LLM ─────────────────────────────────────────────────────────────────
# Gemini model id. Cheapest first (per content/how-to/GEMINI_ERROR_HANDLING_SKILL.md):
#   'gemini-2.5-flash-lite'  — cheapest stable (DEFAULT); fine for one-shot classification
#   'gemini-2.5-flash'       — standard, more capable (thinking ON by default — see DISABLE_THINKING)
#   'gemini-2.5-pro'         — most capable; CANNOT disable thinking (min budget 128)
#   'gemini-2.0-flash-lite' / 'gemini-2.0-flash' — DEPRECATED (EOL ~June 2026)
MODEL_NAME  = 'gemini-2.5-flash-lite'
TEMPERATURE = 0.0         # deterministic classification; raise only if you want sampling diversity

# Thinking is opt-out on gemini-2.5+ models (it inflates token cost a lot for tasks that
# don't need step-by-step reasoning). For one-shot classification we want it OFF; the
# config built below only attaches a ThinkingConfig when the chosen model is in the 2.5+
# family. Set False ONLY if you deliberately want the model to reason before answering.
DISABLE_THINKING = True

# ── Retry / backoff ─────────────────────────────────────────────────────
MAX_RETRIES     = 3
INITIAL_BACKOFF = 2.0     # seconds; doubled on each retry

# ── I/O ─────────────────────────────────────────────────────────────────
INPUT_PATH        = partitioned_folder / 'llm_bootstrap_dataset.pkl'
OUTPUT_CSV        = hitl_folder / 'llm_bootstrap_labels.csv'
OUTPUT_PKL        = hitl_folder / 'llm_bootstrap_labels_full.pkl'
CHECKPOINT_PREFIX = 'llm_bootstrap_checkpoint'
CHECKPOINT_EVERY  = 1_000  # save partial results every N tweets

## Categories and Criteria

**TODO — fill these in before running.**

- `CATEGORIES` is the closed list of allowed labels. The LLM must return one of these strings.
- `CATEGORY_CRITERIA` is the per-category description that goes into the prompt.
  Be precise: include a definition, 2-3 positive examples, and 1-2 exclusions per category.

In [ ]:
# TODO: list every allowed category label exactly as you want it written in the output CSV.
CATEGORIES: list[str] = [
    # 'category_a',
    # 'category_b',
    # 'category_c',
]

# TODO: write the full criteria for each category. The LLM sees this verbatim.
CATEGORY_CRITERIA: str = """
<<< FILL IN THE PER-CATEGORY CRITERIA HERE >>>
""".strip()

assert CATEGORIES, 'CATEGORIES is empty — fill it in before running.'
assert '<<< FILL IN' not in CATEGORY_CRITERIA, 'CATEGORY_CRITERIA still contains the placeholder.'

## API Key

**TODO — provide a Gemini API key before running.**

On Colab, store the key as a notebook secret named `GEMINI_API_KEY` (left sidebar → key icon)
and the cell below picks it up via `userdata.get`. Locally, set the `GEMINI_API_KEY`
environment variable. **Never hard-code the key in the notebook.**

In [ ]:
API_KEY = ''  # leave empty — populated below from Colab secrets / env var

if not RUNNING_LOCALLY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception as e:
        print(f'Colab userdata lookup failed: {e}')
else:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, 'GEMINI_API_KEY not set. Add it as a Colab secret or env var before running.'
assert MODEL_NAME, 'MODEL_NAME is empty — set it in the Configuration cell.'

client = genai.Client(api_key=API_KEY)

config_kwargs = dict(
    temperature=TEMPERATURE,
    response_mime_type='application/json',
)
if MODEL_NAME.startswith('gemini-2.5-pro'):
    # gemini-2.5-pro cannot turn thinking off; minimum thinking_budget is 128.
    config_kwargs['thinking_config'] = types.ThinkingConfig(thinking_budget=128)
    print(f'Note: {MODEL_NAME} cannot disable thinking; pinning thinking_budget=128')
elif DISABLE_THINKING and MODEL_NAME.startswith('gemini-2.5'):
    config_kwargs['thinking_config'] = types.ThinkingConfig(thinking_budget=0)
    print(f'Thinking disabled for {MODEL_NAME} (DISABLE_THINKING=True)')
elif not DISABLE_THINKING and MODEL_NAME.startswith('gemini-2.5'):
    print(f'WARNING: thinking is ENABLED for {MODEL_NAME} — expect higher token cost')
GEN_CONFIG = types.GenerateContentConfig(**config_kwargs)

print(f'LLM client ready: {MODEL_NAME}')

## Prompt and Response Schema

The LLM is asked to return a strict JSON object:
```
{"label": "<one of CATEGORIES>", "confidence": <float 0-1>, "rationale": "<one short sentence>"}
```
Anything else is treated as a parse error: it is retried, and on final failure the row is marked `PARSE_ERROR`.

In [ ]:
def build_prompt(tweet_text: str) -> str:
    return (
        'You are a tweet classifier for a research project on AI public trust.\n'
        'Classify the tweet into exactly one of the following categories:\n'
        f'{", ".join(CATEGORIES)}\n\n'
        'Per-category criteria:\n'
        f'{CATEGORY_CRITERIA}\n\n'
        'Return ONLY a JSON object with this exact schema (no prose, no markdown fences):\n'
        '{"label": "<one of the categories above>", '
        '"confidence": <number between 0 and 1>, '
        '"rationale": "<one short sentence>"}\n\n'
        f'Tweet:\n"""{tweet_text}"""'
    )

## Classification Function

Single-tweet wrapper: build prompt → call LLM → parse + validate JSON → retry on transient errors.

In [ ]:
PARSE_ERROR_RESULT = {'label': 'PARSE_ERROR', 'confidence': 0.0, 'rationale': ''}

def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
    return raw.strip()

def classify_tweet(text: str) -> dict:
    prompt = build_prompt(text)
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=GEN_CONFIG,
            )
            raw  = _strip_code_fences(resp.text or '')
            parsed = json.loads(raw)
            label = str(parsed.get('label', '')).strip()
            if label not in CATEGORIES:
                raise ValueError(f'label {label!r} not in CATEGORIES')
            return {
                'label': label,
                'confidence': float(parsed.get('confidence', 0.0)),
                'rationale': str(parsed.get('rationale', ''))[:500],
            }
        except Exception as e:
            last_error = f'{type(e).__name__}: {e}'
            if attempt + 1 < MAX_RETRIES:
                time.sleep(INITIAL_BACKOFF * (2 ** attempt))
    return {**PARSE_ERROR_RESULT, 'rationale': last_error[:500]}

## Load Input

In [ ]:
assert INPUT_PATH.exists(), f'Input not found: {INPUT_PATH}. Run 00_hitl_data_preparation.ipynb first.'
df = pd.read_pickle(INPUT_PATH)
print(f'Loaded {len(df):,} tweets from {INPUT_PATH.name}')

if SMOKE_TEST:
    df = df.sample(n=min(SMOKE_TEST_N, len(df)), random_state=42).reset_index(drop=True)
    print(f'SMOKE_TEST mode → using {len(df)} tweets')

for col in ('id', 'text', 'likes', 'retweets'):
    if col not in df.columns:
        df[col] = '' if col in ('id', 'text') else 0

df['text'] = df['text'].astype(str)

## Run Classification with Checkpointing

In [ ]:
results: list[dict] = []
t0 = time.time()

for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc='LLM labelling'):
    classification = classify_tweet(row['text'])
    results.append({
        'id': row['id'],
        'text': row['text'],
        'likes': row.get('likes', 0),
        'retweets': row.get('retweets', 0),
        'predicted_label': classification['label'],
        'confidence': classification['confidence'],
        'rationale': classification['rationale'],
        'human_label': '',
    })
    if (len(results) % CHECKPOINT_EVERY) == 0:
        ckpt = hitl_folder / f'{CHECKPOINT_PREFIX}_{len(results)}.pkl'
        pd.DataFrame(results).to_pickle(ckpt)
        tqdm.tqdm.write(f'checkpoint → {ckpt.name} ({time.time()-t0:.0f}s elapsed)')

out_df = pd.DataFrame(results)
print(f'Done. Total time: {time.time()-t0:.1f}s')
print(out_df['predicted_label'].value_counts(dropna=False))

## Save Output

Two artifacts:
- **`llm_bootstrap_labels.csv`** — same schema as `hitl_review_batch_*.csv`, ready to drop into `02_hitl_training_loop.ipynb`.
- **`llm_bootstrap_labels_full.pkl`** — same data **plus** `confidence` and `rationale` columns for inspection.

In [ ]:
hitl_schema_cols = ['id', 'text', 'likes', 'retweets', 'predicted_label', 'human_label']
out_df[hitl_schema_cols].to_csv(OUTPUT_CSV, index=False)
out_df.to_pickle(OUTPUT_PKL)

n_errors = (out_df['predicted_label'] == 'PARSE_ERROR').sum()
print(f'Saved → {OUTPUT_CSV}')
print(f'Saved → {OUTPUT_PKL}')
print(f'PARSE_ERROR rows: {n_errors:,} / {len(out_df):,} ({n_errors/max(len(out_df),1):.1%})')